<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/miso_dir_mnist_qp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
import time

# ── HELPERS classification ────────────────────────────────────────────────
def cls_acc(f, y):
    sg = np.sign(f)
    return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
def cls_auc(f, y):
    try: return float(roc_auc_score((y > 0).astype(int), f))
    except Exception: return float('nan')
def cls_str(f, y):
    return f"acc={cls_acc(f,y):.4f} AUC={cls_auc(f,y):.4f}"

# ══════════════════════════════════════════════════════════════════════════════
# HYPERPARAMÈTRES — une section par type d'expert + une pour la Phase 2.
# n_dirs / batch_dirs / k_loss PARTAGÉS (mêmes directions Monte-Carlo pour tous
# les experts — nécessaire pour que les termes croisés de la Phase 2 aient un
# sens).
# ══════════════════════════════════════════════════════════════════════════════
params_shared = {
    "neg_digits": [3, 5], "pos_digits": [8], "img_size": 7,
    "seed": 187225,
    "n_train": 300, "n_test": 5000,
    "n_dirs": 50, "batch_dirs": 10,
    "k_loss": 3000000,
    "deg_P": 3,   # degré du Ridge polynomial de comparaison
}
params_shared["n_unlabeled"] = np.maximum(4000 - params_shared["n_train"], 500)
params_shared["train_center_ratio"] = 0.5 + params_shared["n_train"] / (2 * 4000)

params_gauss = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "lambda_reg": 5, "thresh_factor": 1e-2,
    "sigma_min": 20, "sigma_max": 30,
    "n_dict": 2000, "n_centres": 800,
    "n_G": 1000,
    "n_experts": 2,
}
# semi-interpolation (QP Sobolev) — remplace Ridge pour gauss & wnd
params_qp = {
    "qp_margin":  1.0,    # y_i * f(x_i) >= margin
    "const_pen":  1e-5,   # pénalité sur le biais
    "lambda_G":   1e-9,   # régularisation numérique de G
    "thres1":     1e-8,   # seuil bas spectral
    "thres2":     1e12,   # seuil haut spectral (optionnel)
}
params_wnd = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "lambda_reg": 5, "thresh_factor": 1e-2,
    "sigma_min":15 , "sigma_max": 20,
    "n_dict": 2000, "n_centres": 1000,
    "n_G": 1000,
    "n_experts": 0,
}

params = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},#param_greedy
    "levels": 9,
    "sigma0": 10.0,
    "sigma_decay": 0.8,
    "n_dict_candidates": 2000,
    "n_centers_per_level": 800,
    "n_stop": 1e3,
    "plateau_window": 14,
    "plateau_tol": 1e-6,
    "lambda_reg": 2,
    "thres_factor": 1e-4,
    "n_G": 2000,
}

params_phase2 = {
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "lambda_reg": 0.1, "thresh_factor": 1e-3,
    "n_G_H": 2000,
}

# ══════════════════════════════════════════════════════════════════════════════
# DONNÉES : MNIST, chiffres neg_digits vs pos_digits, réduits 28×28 -> img_size²
# ══════════════════════════════════════════════════════════════════════════════
from tensorflow.keras.datasets import mnist

def load_mnist_binary(neg_list, pos_list, img_size, n_train, n_test, n_unlabeled, seed):
    (Xtr, ytr), (Xte, yte) = mnist.load_data()
    X = np.concatenate([Xtr, Xte], axis=0).astype(float)
    yraw = np.concatenate([ytr, yte])

    mask = np.isin(yraw, neg_list + pos_list)
    X = X[mask]; yraw = yraw[mask]
    y = np.where(np.isin(yraw, pos_list), 1., -1.)

    b = 28 // img_size
    X = X.reshape(-1, img_size, b, img_size, b).mean(axis=(2, 4))
    X = X.reshape(len(X), img_size*img_size)
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)

    rng = np.random.default_rng(seed)
    neg = np.where(y < 0)[0]; pos = np.where(y > 0)[0]
    rng.shuffle(neg); rng.shuffle(pos)

    ntr = n_train // 2; nte = n_test // 2
    itr = np.concatenate([neg[:ntr], pos[:ntr]])
    ite = np.concatenate([neg[ntr:ntr+nte], pos[ntr:ntr+nte]])
    used = set(itr) | set(ite)
    pool = np.array([i for i in range(len(X)) if i not in used])
    iu = rng.choice(pool, n_unlabeled, replace=False)

    rng.shuffle(itr); rng.shuffle(ite); rng.shuffle(iu)
    return X[itr], y[itr], X[ite], y[ite], X[iu]

X_train, y_train, X_test, y_test, X_unlabeled = load_mnist_binary(
    params_shared["neg_digits"], params_shared["pos_digits"], params_shared["img_size"],
    params_shared["n_train"], params_shared["n_test"], params_shared["n_unlabeled"],
    seed=params_shared["seed"])

X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)
d = X_train.shape[1]

print(f"digits {params_shared['neg_digits']} vs {params_shared['pos_digits']}, "
      f"{params_shared['img_size']}×{params_shared['img_size']} -> dim {d}")
print(f"  train: {len(X_train)}  test: {len(X_test)}  unlabeled: {len(X_unlabeled)}")
print(f"  train (+1) = {(y_train>0).sum()}/{len(y_train)}")
print(f"  test  (+1) = {(y_test>0).sum()}/{len(y_test)}")
_bal_te = (y_test > 0).mean()
if _bal_te < 0.3 or _bal_te > 0.7:
    print(f"  [!] classes déséquilibrées ({_bal_te:.1%} de +1) — préférer l'AUC.")


def sample_candidates_aniso(X_train, X_unlabeled, n_candidates, train_ratio, rng, dd,
                            sigma_min, sigma_max):
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates-n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    centers = np.vstack(parts)
    log_s = rng.uniform(np.log(sigma_min), np.log(sigma_max), size=(len(centers), dd))
    sigmas = np.exp(log_s)
    return centers, sigmas

def sample_candidates_plain(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates-n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    return np.vstack(parts)


# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES ANISOTROPES DIRECTIONNELLES — un σ par dimension par centre.
# phi(x)=exp(-Σ(x_i-c_i)²/(2σ_i²)).
# ══════════════════════════════════════════════════════════════════════════════
def gaussian_features_aniso(X, centers, sigmas):
    diff = X[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2/sigmas[None, :, :]**2, axis=2)
    return np.exp(-sq/2)

def build_G_gauss_aniso_directional_streaming(X_cloud, centers, sigmas, weights, n_dirs,
                                              batch_dirs=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]
    inv2 = 1.0/sigmas**2
    sq = np.sum(diff**2*inv2[None, :, :], axis=2)
    phi = np.exp(-sq/2)

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0*(phi.T@phi)/n
    if need2:
        TR = (-np.sum(inv2, axis=1)[None, :] + np.sum(diff**2*inv2[None, :, :]**2, axis=2))*phi
        TRG = (TR.T@TR)/n
    if need3:
        Csum = -np.sum(inv2, axis=1)
        Dq = np.sum(diff**2*inv2[None, :, :]**2, axis=2)
        coefV = 2*inv2[None, :, :]**2 - (Csum[None, :, None]+Dq[:, :, None])*inv2[None, :, :]
        V = phi[:, :, None]*diff*coefV
        VVG = np.einsum('nid,njd->ij', V, V)/n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
        B = np.einsum('nkd,md->nmk', U**2, inv2)
        gp = -A; gpp = -B
        if w1:
            D1 = gp*phi[:, :, None]
            D1f = D1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T@D1f
            del D1, D1f
        if need2 or need3:
            D2 = (gpp+gp**2)*phi[:, :, None]
            if need2:
                D2f = D2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T@D2f
                del D2f
            if need3:
                D3 = (3*gp*gpp+gp**3)*phi[:, :, None]
                D3f = D3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T@D3f
                del D3, D3f
            del D2
        del U, A, B, gp, gpp
        done += K

    if w1: G += w1*d_*MC1_sum/(n*n_dirs)
    if need2:
        MC2 = MC2_sum/(n*n_dirs)
        G += w2*(d_*(d_+2)*MC2 - TRG)/2
    if need3:
        MC3 = MC3_sum/(n*n_dirs)
        G += w3*(d_*(d_+2)*(d_+4)*MC3 - 9*VVG)/6
    return G


# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES ISOTROPES DIRECTIONNELLES ("miso") — UN σ GLOBAL par niveau
# (partagé par tous les centres de ce niveau, PAS un σ aléatoire par centre).
# phi(x)=exp(-‖x-c‖²/(2σ²)).
# ══════════════════════════════════════════════════════════════════════════════
def gaussian_features_iso(X, centers, sigma):
    diff = X[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2, axis=2)
    return np.exp(-sq/(2*sigma**2))

def build_G_miso_directional_streaming(X_cloud, centers, sigma, weights, n_dirs,
                                       batch_dirs=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2, axis=2)
    phi = np.exp(-sq/(2*sigma**2))

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0*(phi.T@phi)/n
    if need2:
        TR = (-d_/sigma**2 + sq/sigma**4)*phi
        TRG = (TR.T@TR)/n
    if need3:
        A = -d_/sigma**2; B = 1/sigma**4
        coefV = 2*B - (A/sigma**2) - (B/sigma**2)*sq
        V = phi[:, :, None]*diff*coefV[:, :, None]
        VVG = np.einsum('nid,njd->ij', V, V)/n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        a = np.einsum('nmd,nkd->nmk', diff, U)
        gp = -a/sigma**2
        if w1:
            D1 = gp*phi[:, :, None]
            D1f = D1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T@D1f
            del D1, D1f
        if need2 or need3:
            gpp = -1/sigma**2
            D2 = (gpp + gp**2)*phi[:, :, None]
            if need2:
                D2f = D2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T@D2f
                del D2f
            if need3:
                D3 = (3*gp*gpp + gp**3)*phi[:, :, None]
                D3f = D3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T@D3f
                del D3, D3f
            del D2
        del U, a, gp
        done += K

    if w1: G += w1*d_*MC1_sum/(n*n_dirs)
    if need2:
        MC2 = MC2_sum/(n*n_dirs)
        G += w2*(d_*(d_+2)*MC2 - TRG)/2
    if need3:
        MC3 = MC3_sum/(n*n_dirs)
        G += w3*(d_*(d_+2)*(d_+4)*MC3 - 9*VVG)/6
    return G


# ══════════════════════════════════════════════════════════════════════════════
# WENDLAND ANISOTROPES DIRECTIONNELS — φ(r), r=‖(x-c)/σ‖.
# Profil choisi selon l'ordre max requis par params_wnd["weights"] OU
# params_phase2["weights"] (Phase 2 doit pouvoir dériver les experts Wendland
# à son propre ordre).
# ══════════════════════════════════════════════════════════════════════════════
def _wnd_raw_profile(max_order):
    k = max(1, int(np.ceil(max_order/2)))
    if k == 1:      # C²
        return (lambda r,rp: rp**4*(4*r+1),
                lambda r,rp: -20*r*rp**3,
                lambda r,rp: 20*rp**2*(4*r-1),
                lambda r,rp: 120*rp*(1-2*r))
    elif k == 2:    # C⁴
        return (lambda r,rp: rp**6*(35*r**2+18*r+3)/3,
                lambda r,rp: -56*r*rp**5*(5*r+1)/3,
                lambda r,rp: 56*rp**4*(35*r**2-4*r-1)/3,
                lambda r,rp: -560*r*rp**3*(7*r-3))
    else:           # C⁶
        return (lambda r,rp: rp**8*(32*r**3+25*r**2+8*r+1),
                lambda r,rp: -22*r*rp**7*(16*r**2+7*r+1),
                lambda r,rp: 22*rp**6*(160*r**3+15*r**2-6*r-1),
                lambda r,rp: -1584*r*rp**5*(20*r**2-5*r-1))

_wnd_orders_all = ([o for o, w in params_wnd["weights"].items() if w != 0] +
                   [o for o, w in params_phase2["weights"].items() if w != 0])
_wnd_max_order = max(_wnd_orders_all) if _wnd_orders_all else 1
WND_PHI, WND_P1, WND_P2, WND_P3 = _wnd_raw_profile(_wnd_max_order)
print(f"Profil Wendland C{2*max(1,int(np.ceil(_wnd_max_order/2)))} "
      f"(ordre max requis, Phase 1 wnd ∪ Phase 2 = {_wnd_max_order})")

def wnd_features_aniso(X, centers, sigmas):
    diff = X[:, None, :] - centers[None, :, :]
    inv2 = 1.0/sigmas**2
    r = np.sqrt(np.sum(diff**2*inv2[None, :, :], axis=2))
    rp = np.maximum(1.-r, 0.)
    return WND_PHI(r, rp)

def build_G_wnd_directional_streaming(X_cloud, centers, sigmas, weights, n_dirs,
                                      batch_dirs=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]
    inv2 = 1.0/sigmas**2
    r = np.sqrt(np.sum(diff**2*inv2[None, :, :], axis=2))
    rp = np.maximum(1.-r, 0.)
    r_safe = np.maximum(r, 1e-9)
    p0, p1v, p2v, p3v = WND_PHI(r, rp), WND_P1(r, rp), WND_P2(r, rp), WND_P3(r, rp)

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0*(p0.T@p0)/n

    if need2 or need3:
        Q4 = np.sum(diff**2*inv2[None, :, :]**2, axis=2)
        S0 = np.sum(inv2, axis=1)
    if need2:
        TR = p2v*Q4/r_safe**2 + p1v*(S0[None, :]/r_safe - Q4/r_safe**3)
        TR = np.where(r > 1e-9, TR, 0.)
        TRG = (TR.T@TR)/n
    if need3:
        dr = diff*inv2[None, :, :]/r_safe[:, :, None]
        dQ4 = 2*diff*inv2[None, :, :]**2
        V = (p3v[:, :, None]*dr*Q4[:, :, None]/r_safe[:, :, None]**2
            + p2v[:, :, None]*(dQ4/r_safe[:, :, None]**2 - 2*Q4[:, :, None]*dr/r_safe[:, :, None]**3)
            + p2v[:, :, None]*dr*(S0[None, :, None]/r_safe[:, :, None] - Q4[:, :, None]/r_safe[:, :, None]**3)
            + p1v[:, :, None]*(-S0[None, :, None]*dr/r_safe[:, :, None]**2
                               - dQ4/r_safe[:, :, None]**3
                               + 3*Q4[:, :, None]*dr/r_safe[:, :, None]**4))
        V = np.where(r[:, :, None] > 1e-9, V, 0.)
        VVG = np.einsum('nid,njd->ij', V, V)/n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
        B = np.einsum('nkd,md->nmk', U**2, inv2)
        r0 = r_safe[:, :, None]
        rprime = A/r0
        rpprime = -A**2/r0**3 + B/r0
        f1 = p1v[:, :, None]*rprime
        if w1:
            D1f = f1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T@D1f
            del D1f
        if need2 or need3:
            f2 = p2v[:, :, None]*rprime**2 + p1v[:, :, None]*rpprime
            if need2:
                D2f = f2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T@D2f
                del D2f
            if need3:
                rppprime = 3*A*(A**2 - B*r0**2)/r0**5
                f3 = p3v[:, :, None]*rprime**3 + 3*p2v[:, :, None]*rprime*rpprime + p1v[:, :, None]*rppprime
                D3f = f3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T@D3f
                del D3f, f3, rppprime
            del f2
        del U, A, B, rprime, rpprime, f1
        done += K

    if w1: G += w1*d_*MC1_sum/(n*n_dirs)
    if need2:
        MC2 = MC2_sum/(n*n_dirs)
        G += w2*(d_*(d_+2)*MC2 - TRG)/2
    if need3:
        MC3 = MC3_sum/(n*n_dirs)
        G += w3*(d_*(d_+2)*(d_+4)*MC3 - 9*VVG)/6
    return G



def gaussian_features(X, centers, sigma):
    diff = X[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2, axis=2)
    return np.exp(-sq/(2*sigma**2))
##===================================
#solve qp
#========================================
from scipy.optimize import minimize, LinearConstraint

def solve_qp(A, y, G, margin):
    """min c^T G c  s.t.  y_i (A c)_i >= margin."""
    n, k = A.shape
    Gr = G + 1e-12 * np.eye(k)
    con = LinearConstraint(np.diag(y) @ A, lb=margin, ub=np.inf)
    try:
        c0 = np.linalg.lstsq(A, 1.5 * margin * y, rcond=None)[0]
    except Exception:
        c0 = np.zeros(k)
    res = minimize(
        lambda c: c @ Gr @ c,
        c0,
        jac=lambda c: 2 * Gr @ c,
        constraints=[con],
        method="SLSQP",
        options={"maxiter": 800, "ftol": 1e-11},
    )
    c = res.x
    marge_eff = float(np.min(y * (A @ c)))
    return c, marge_eff >= margin - 1e-4, marge_eff


def fit_qp_rbf(A, Atest, y_train, y_test, G, qp_margin, const_pen,
               lambda_G, thres1, thres2, titre=""):
    """
    Semi-interpolation Sobolev :
      min ‖u‖_H² (+ pénalité biais)  s.t.  y_i f(x_i) >= qp_margin
    """
    n, m = A.shape
    G_reg = G + lambda_G * np.eye(m)

    s_g, V_g = np.linalg.eigh(G_reg)
    keep = (s_g > thres1) & (s_g < thres2)
    if keep.sum() == 0:
        raise ValueError(f"[{titre}] bande spectrale vide")
    T = V_g[:, keep] / np.sqrt(s_g[keep])
    r = int(keep.sum())

    mean_phi = A.mean(axis=0)
    A_c = A - mean_phi

    A_qp = np.hstack([A_c @ T, np.ones((n, 1))])
    G_qp = np.eye(r + 1)
    G_qp[-1, -1] = const_pen

    sol, feas, marge = solve_qp(A_qp, y_train, G_qp, qp_margin)
    coef = T @ sol[:r]
    off = sol[-1] - mean_phi @ coef

    f_tr = A @ coef + off
    f_te = Atest @ coef + off
    normH = float(np.sqrt(max(sol[:r] @ sol[:r], 0.0)))
    mse_te = float(np.mean((f_te - y_test) ** 2))

    print(
        f"  [{titre}] n_feat={m} rang={r} | faisable={feas} marge={marge:.3f} "
        f"‖u‖_H={normH:.4f} MSE_te={mse_te:.4f} | {cls_str(f_te, y_test)}"
    )
    return {
        "coef": coef,
        "off": off,
        "f_tr": f_tr,
        "f_te": f_te,
        "normH": normH,
        "feas": feas,
        "marge": marge,
        "mse_te": mse_te,
    }
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : experts gaussiens anisotropes (semi-interpolation QP)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 1 (gaussien anisotrope, semi-interp) : "
      f"{params_gauss['n_experts']} experts indépendants...")
experts = []
F_train_list = []
F_test_list = []
times_p1 = []

for e in range(params_gauss["n_experts"]):
    t0 = time.time()
    rng = np.random.default_rng(seed=100 + e)

    candidates, sigmas_cand = sample_candidates_aniso(
        X_train, X_unlabeled, params_gauss["n_dict"],
        params_shared["train_center_ratio"], rng, d,
        params_gauss["sigma_min"], params_gauss["sigma_max"])
    A_cand = gaussian_features_aniso(X_train, candidates, sigmas_cand)
    corr = (A_cand.T @ y_train) / len(X_train)
    scores = corr**2
    k = min(params_gauss["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers = candidates[top_k]
    sigmas_sel = sigmas_cand[top_k]
    A = A_cand[:, top_k]
    Atest = gaussian_features_aniso(X_test, centers, sigmas_sel)

    n_G = min(N_all, params_gauss["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G_gauss_aniso_directional_streaming(
        X_all[idx_G], centers, sigmas_sel, params_gauss["weights"],
        n_dirs=params_shared["n_dirs"], batch_dirs=params_shared["batch_dirs"],
        rng=np.random.default_rng(9000 + e))

    sol = fit_qp_rbf(
        A, Atest, y_train, y_test, G,
        qp_margin=params_qp["qp_margin"],
        const_pen=params_qp["const_pen"],
        lambda_G=params_qp["lambda_G"],
        thres1=params_qp["thres1"],
        thres2=params_qp["thres2"],
        titre=f"gauss {e+1}/{params_gauss['n_experts']}",
    )
    coeffs, off = sol["coef"], sol["off"]
    f_tr, f_te = sol["f_tr"], sol["f_te"]

    t1 = time.time()
    times_p1.append(t1 - t0)
    F_train_list.append(f_tr)
    F_test_list.append(f_te)
    experts.append({
        "type": "gauss",
        "centers": centers,
        "sigmas": sigmas_sel,
        "coeffs": coeffs,
        "off": off,
    })
    print(f" gauss {e+1}/{params_gauss['n_experts']} | "
          f"{cls_str(f_te, y_test)} | {times_p1[-1]:.1f}s")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : experts Wendland anisotropes (semi-interpolation QP)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 1 (Wendland, semi-interp) : "
      f"{params_wnd['n_experts']} experts indépendants...")

for e in range(params_wnd["n_experts"]):
    t0 = time.time()
    rng = np.random.default_rng(seed=500 + e)

    candidates, sigmas_cand = sample_candidates_aniso(
        X_train, X_unlabeled, params_wnd["n_dict"],
        params_shared["train_center_ratio"], rng, d,
        params_wnd["sigma_min"], params_wnd["sigma_max"])
    A_cand = wnd_features_aniso(X_train, candidates, sigmas_cand)
    corr = (A_cand.T @ y_train) / len(X_train)
    scores = corr**2
    k = min(params_wnd["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers = candidates[top_k]
    sigmas_sel = sigmas_cand[top_k]
    A = A_cand[:, top_k]
    Atest = wnd_features_aniso(X_test, centers, sigmas_sel)

    n_G = min(N_all, params_wnd["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G_wnd_directional_streaming(
        X_all[idx_G], centers, sigmas_sel, params_wnd["weights"],
        n_dirs=params_shared["n_dirs"], batch_dirs=params_shared["batch_dirs"],
        rng=np.random.default_rng(9500 + e))

    sol = fit_qp_rbf(
        A, Atest, y_train, y_test, G,
        qp_margin=params_qp["qp_margin"],
        const_pen=params_qp["const_pen"],
        lambda_G=params_qp["lambda_G"],
        thres1=params_qp["thres1"],
        thres2=params_qp["thres2"],
        titre=f"wnd {e+1}/{params_wnd['n_experts']}",
    )
    coeffs, off = sol["coef"], sol["off"]
    f_tr, f_te = sol["f_tr"], sol["f_te"]

    t1 = time.time()
    times_p1.append(t1 - t0)
    if np.max(np.abs(f_tr)) < 1e-12:
        print(f" [!] wnd {e+1} quasi nul sur le train (support compact non couvert)")
    F_train_list.append(f_tr)
    F_test_list.append(f_te)
    experts.append({
        "type": "wnd",
        "centers": centers,
        "sigmas": sigmas_sel,
        "coeffs": coeffs,
        "off": off,
    })
    print(f" wnd {e+1}/{params_wnd['n_experts']} | "
          f"{cls_str(f_te, y_test)} | {times_p1[-1]:.1f}s")




#___________________________________dodatki_gpt

def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr = min(
        int(round(n_candidates * train_ratio)),
        X_train.shape[0]
    )

    n_ul = min(
        n_candidates - n_tr,
        X_unlabeled.shape[0]
    )

    parts = []

    if n_tr > 0:
        parts.append(
            X_train[
                rng.choice(
                    X_train.shape[0],
                    n_tr,
                    replace=False
                )
            ]
        )

    if n_ul > 0:
        parts.append(
            X_unlabeled[
                rng.choice(
                    X_unlabeled.shape[0],
                    n_ul,
                    replace=False
                )
            ]
        )

    return np.vstack(parts)


#______________________
#______________dodatki2_gpt(construction des features pour le glouton)

def build_G_directional_streaming(X_cloud, centers, sigma, weights,
                                  n_dirs, batch_dirs=10, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    n, d_ = X_cloud.shape
    m = centers.shape[0]

    diff = X_cloud[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2, axis=2)
    phi = np.exp(-sq/(2*sigma**2))

    w0 = weights.get(0, 0.)
    w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.)
    w3 = weights.get(3, 0.)

    need2 = w2 != 0
    need3 = w3 != 0

    G = np.zeros((m, m))

    if w0:
        G += w0 * (phi.T @ phi) / n

    if need2:
        TR = (-d_/sigma**2 + sq/sigma**4) * phi
        TRG = (TR.T @ TR) / n

    if need3:
        A = -d_/sigma**2
        B = 1/sigma**4
        coefV = 2*B - (A/sigma**2) - (B/sigma**2)*sq
        V = phi[:, :, None] * diff * coefV[:, :, None]
        VVG = np.einsum('nid,njd->ij', V, V) / n

    MC1_sum = np.zeros((m, m))
    MC2_sum = np.zeros((m, m))
    MC3_sum = np.zeros((m, m))

    done = 0

    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)

        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)

        a = np.einsum('nmd,nkd->nmk', diff, U)

        gp = -a/sigma**2

        if w1:
            D1 = gp * phi[:, :, None]
            D1f = D1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T @ D1f
            del D1, D1f

        if need2 or need3:
            gpp = -1/sigma**2

            D2 = (gpp + gp**2) * phi[:, :, None]

            if need2:
                D2f = D2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T @ D2f
                del D2f

            if need3:
                D3 = (3*gp*gpp + gp**3) * phi[:, :, None]
                D3f = D3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T @ D3f
                del D3, D3f

            del D2

        del U, a, gp
        done += K

    if w1:
        G += w1 * d_ * MC1_sum / (n*n_dirs)

    if need2:
        MC2 = MC2_sum / (n*n_dirs)
        G += w2 * (d_*(d_+2)*MC2 - TRG) / 2

    if need3:
        MC3 = MC3_sum / (n*n_dirs)
        G += w3 * (d_*(d_+2)*(d_+4)*MC3 - 9*VVG) / 6

    return G



# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : EXPERT GLOUTON
#
# La cascade gloutonne complète constitue UN SEUL expert.
# Chaque niveau ajoute sa correction au résidu du niveau précédent.
# À la fin, seul le niveau final est conservé comme fonction de l'expert.
# ══════════════════════════════════════════════════════════════════════════════

print(
    f"\nPhase 1 (glouton) : 1 expert "
    f"(levels={params['levels']}, "
    f"σ0={params['sigma0']}, decay={params['sigma_decay']})..."
)

t0_expert = time.time()

# Prédictions cumulées de la cascade
pred_train_accum = np.zeros(len(X_train))
pred_test_accum = np.zeros(len(X_test))

losses_g = []
history_g = []

stop_reason_g = None
final_level_g = None

for level in range(params["levels"]):

    t0 = time.time()

    sigma = params["sigma0"] * params["sigma_decay"]**level
    lambda_reg = max(params["lambda_reg"], 0)

    rng = np.random.default_rng(seed=level)

    # ── dictionnaire de centres ─────────────────────────────────────────────
    candidates = sample_candidates(
        X_train,
        X_unlabeled,
        params["n_dict_candidates"],
        params_shared["train_center_ratio"],
        rng
    )

    A_cand = gaussian_features(
        X_train,
        candidates,
        sigma
    )

    # Résidu de la cascade précédente
    residual = y_train - pred_train_accum

    # Sélection gloutonne des centres
    corr = (A_cand.T @ residual) / len(X_train)
    scores = corr**2

    k = min(
        params["n_centers_per_level"],
        len(candidates)
    )

    top_k = np.argsort(scores)[-k:]

    centers = candidates[top_k]

    A = A_cand[:, top_k]

    Atest = gaussian_features(
        X_test,
        centers,
        sigma
    )

    # ── Gram Sobolev directionnel ───────────────────────────────────────────
    n_G = min(N_all, params["n_G"])

    idx_G = rng.choice(
        N_all,
        size=n_G,
        replace=False
    )

    G = build_G_directional_streaming(
        X_all[idx_G],
        centers,
        sigma,
        params["weights"],
        n_dirs=params_shared["n_dirs"],
        batch_dirs=params_shared["batch_dirs"],
        rng=np.random.default_rng(9000 + level)
    )

    # ── résolution régularisée sur le résidu ────────────────────────────────
    n = A.shape[0]

    M = (A.T @ A) / n + lambda_reg * G
    rhs = (A.T @ residual) / n

    eigvals, eigvecs = np.linalg.eigh(M)

    thresh = max(
        lambda_reg * params["thres_factor"],
        1e-7
    )

    mask = eigvals > thresh

    V = eigvecs[:, mask]
    S = eigvals[mask]

    coeffs = V @ ((V.T @ rhs) / S)

    # Loss du niveau
    loss_data = np.mean(
        (residual - A @ coeffs)**2
    )

    loss_reg = (
        lambda_reg *
        coeffs @ G @ coeffs
    )

    loss_total = loss_data + loss_reg

    # ── nouvelle fonction cumulée de la cascade ─────────────────────────────
    pred_train_new = pred_train_accum + A @ coeffs
    pred_test_new = pred_test_accum + Atest @ coeffs

    elapsed = time.time() - t0

    print(
        f"\n  glouton niveau {level+1}/{params['levels']} "
        f"| σ={sigma:.4f} "
        f"| rang={mask.sum()}/{k} "
        f"| loss={loss_total:.6f} "
        f"(data={loss_data:.6f} reg={loss_reg:.6f}) "
        f"| {cls_str(pred_test_new, y_test)} "
        f"| {elapsed:.1f}s"
    )

    # ── détection d'explosion ───────────────────────────────────────────────
    if len(history_g) > 0:

        prev_loss = history_g[-1]["loss_total"]

        if loss_total > prev_loss + params["n_stop"] * lambda_reg:

            best_idx = len(history_g) - 1

            for i in range(len(history_g) - 2, -1, -1):

                if history_g[i]["loss_total"] <= \
                   history_g[i + 1]["loss_total"]:
                    best_idx = i
                else:
                    break

            print(
                f"  ⚠ ARRÊT : explosion détectée "
                f"-> niveau {history_g[best_idx]['level']+1}"
            )

            stop_reason_g = "explosion"
            final_level_g = best_idx
            break

    # On valide le niveau
    pred_train_accum = pred_train_new
    pred_test_accum = pred_test_new

    losses_g.append(loss_total)

    history_g.append({
        "level": level,
        "pred_train": pred_train_accum.copy(),
        "pred_test": pred_test_accum.copy(),
        "loss_total": loss_total
    })

    # ── détection de plateau ────────────────────────────────────────────────
    w = params["plateau_window"]

    if len(history_g) >= w:

        recent = [
            h["loss_total"]
            for h in history_g[-w:]
        ]

        improvement = recent[0] - recent[-1]

        if improvement < params["plateau_tol"]:

            print(
                f"  ⚠ ARRÊT : plateau détecté "
                f"(baisse sur {w} niveaux = {improvement:.6f})"
            )

            stop_reason_g = "plateau"
            final_level_g = len(history_g) - w
            break


# ── niveau final de la cascade = UN expert ───────────────────────────────────
if stop_reason_g is None:
    final_level_g = len(history_g) - 1
    stop_reason_g = "max_levels"

final_state_g = history_g[final_level_g]

f_tr = final_state_g["pred_train"]
f_te = final_state_g["pred_test"]

elapsed_expert = time.time() - t0_expert

print(
    f"\n  glouton final : niveau "
    f"{final_state_g['level']+1}/{params['levels']} "
    f"| arrêt={stop_reason_g} "
    f"| {cls_str(f_te, y_test)} "
    f"| {elapsed_expert:.1f}s"
)

# ── IMPORTANT : un seul expert est ajouté à la Phase 2 ───────────────────────
F_train_list.append(f_tr)
F_test_list.append(f_te)

experts.append({
    "type": "glouton",
    "centers": centers,
    "sigma": sigma,
    "coeffs": coeffs,
    "level": final_state_g["level"],
    "stop_reason": stop_reason_g
})


# ── reconstruction des matrices utilisées par la Phase 2 ────────────────────
F_train = np.column_stack(F_train_list)
F_test = np.column_stack(F_test_list)

n_exp = len(experts)

_type_counters = {
    "gauss": 0,
    "wnd": 0,
    "miso": 0,
    "glouton": 0
}

expert_names = []

for ex in experts:

    _type_counters[ex["type"]] += 1

    expert_names.append(
        f"{ex['type']}{_type_counters[ex['type']]}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : Gram H entre TOUS les experts + semi-interpolation QP
# directions PARTAGÉES — hyperparamètres INDÉPENDANTS de ceux de Phase 1.
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 (semi-interpolation QP sur les experts)")

rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params_phase2["n_G_H"]), replace=False)
X_H = X_all[idx_H]; n_H = len(idx_H)

w0 = params_phase2["weights"].get(0, 0.); w1 = params_phase2["weights"].get(1, 0.)
w2 = params_phase2["weights"].get(2, 0.); w3 = params_phase2["weights"].get(3, 0.)
need2 = w2 != 0; need3 = w3 != 0

PHI_H = np.zeros((n_H, n_exp))
TR_H  = np.zeros((n_H, n_exp)) if need2 else None
V_H   = np.zeros((n_H, n_exp, d)) if need3 else None

for e, ex in enumerate(experts):
    if ex['type'] in ('gauss', 'wnd'):
        diff = X_H[:, None, :] - ex['centers'][None, :, :]
        inv2 = 1.0 / ex['sigmas']**2
        if ex['type'] == 'gauss':
            sq = np.sum(diff**2 * inv2[None, :, :], axis=2)
            phi_k = np.exp(-sq / 2)
            PHI_H[:, e] = phi_k @ ex['coeffs']
            if need2:
                TR_k = (-np.sum(inv2, axis=1)[None, :] + np.sum(diff**2 * inv2[None, :, :]**2, axis=2)) * phi_k
                TR_H[:, e] = TR_k @ ex['coeffs']
            if need3:
                Csum = -np.sum(inv2, axis=1)
                Dq = np.sum(diff**2 * inv2[None, :, :]**2, axis=2)
                coefV = 2 * inv2[None, :, :]**2 - (Csum[None, :, None] + Dq[:, :, None]) * inv2[None, :, :]
                V_k = phi_k[:, :, None] * diff * coefV
                V_H[:, e, :] = np.einsum('nmd,m->nd', V_k, ex['coeffs'])
        else:  # wnd
            r = np.sqrt(np.sum(diff**2 * inv2[None, :, :], axis=2))
            rp = np.maximum(1. - r, 0.)
            r_safe = np.maximum(r, 1e-9)
            p0k, p1k, p2k = WND_PHI(r, rp), WND_P1(r, rp), WND_P2(r, rp)
            PHI_H[:, e] = p0k @ ex['coeffs']
            if need2 or need3:
                Q4 = np.sum(diff**2 * inv2[None, :, :]**2, axis=2)
                S0 = np.sum(inv2, axis=1)
            if need2:
                TR_k = p2k * Q4 / r_safe**2 + p1k * (S0[None, :] / r_safe - Q4 / r_safe**3)
                TR_k = np.where(r > 1e-9, TR_k, 0.)
                TR_H[:, e] = TR_k @ ex['coeffs']
            if need3:
                p3k = WND_P3(r, rp)
                dr = diff * inv2[None, :, :] / r_safe[:, :, None]
                dQ4 = 2 * diff * inv2[None, :, :]**2
                V_k = (p3k[:, :, None] * dr * Q4[:, :, None] / r_safe[:, :, None]**2
                      + p2k[:, :, None] * (dQ4 / r_safe[:, :, None]**2 - 2 * Q4[:, :, None] * dr / r_safe[:, :, None]**3)
                      + p2k[:, :, None] * dr * (S0[None, :, None] / r_safe[:, :, None] - Q4[:, :, None] / r_safe[:, :, None]**3)
                      + p1k[:, :, None] * (-S0[None, :, None] * dr / r_safe[:, :, None]**2
                                           - dQ4 / r_safe[:, :, None]**3
                                           + 3 * Q4[:, :, None] * dr / r_safe[:, :, None]**4))
                V_k = np.where(r[:, :, None] > 1e-9, V_k, 0.)
                V_H[:, e, :] = np.einsum('nmd,m->nd', V_k, ex['coeffs'])
    else:  # glouton / miso (isotrope)
        sigma = ex['sigma']
        diff = X_H[:, None, :] - ex['centers'][None, :, :]
        sq = np.sum(diff**2, axis=2)
        phi_k = np.exp(-sq / (2 * sigma**2))
        PHI_H[:, e] = phi_k @ ex['coeffs']
        if need2:
            TR_k = (-d / sigma**2 + sq / sigma**4) * phi_k
            TR_H[:, e] = TR_k @ ex['coeffs']
        if need3:
            Am = -d / sigma**2; Bm = 1 / sigma**4
            coefV = 2 * Bm - (Am / sigma**2) - (Bm / sigma**2) * sq
            V_k = phi_k[:, :, None] * diff * coefV[:, :, None]
            V_H[:, e, :] = np.einsum('nmd,m->nd', V_k, ex['coeffs'])

G_H = np.zeros((n_exp, n_exp))
if w0:
    G_H += w0 * (PHI_H.T @ PHI_H) / n_H
if need2:
    TRG_H = (TR_H.T @ TR_H) / n_H
if need3:
    VVG_H = np.einsum('nid,njd->ij', V_H, V_H) / n_H

if w1 or need2 or need3:
    rng_dir = np.random.default_rng(seed=8000)
    MC1_sum = np.zeros((n_exp, n_exp))
    MC2_sum = np.zeros((n_exp, n_exp))
    MC3_sum = np.zeros((n_exp, n_exp))
    done = 0
    while done < params_shared["n_dirs"]:
        K = min(params_shared["batch_dirs"], params_shared["n_dirs"] - done)
        U = rng_dir.normal(size=(n_H, K, d))
        U /= np.linalg.norm(U, axis=2, keepdims=True)

        D1_agg = np.zeros((n_H, K, n_exp)) if w1 else None
        D2_agg = np.zeros((n_H, K, n_exp)) if (need2 or need3) else None
        D3_agg = np.zeros((n_H, K, n_exp)) if need3 else None

        for e, ex in enumerate(experts):
            if ex['type'] in ('gauss', 'wnd'):
                diff = X_H[:, None, :] - ex['centers'][None, :, :]
                inv2 = 1.0 / ex['sigmas']**2
                A = np.einsum('nmd,nkd,md->nmk', diff, U, inv2)
                B = np.einsum('nkd,md->nmk', U**2, inv2)
                if ex['type'] == 'gauss':
                    sq = np.sum(diff**2 * inv2[None, :, :], axis=2)
                    phi_k = np.exp(-sq / 2)
                    gp = -A; gpp = -B
                    if w1:
                        D1_k = gp * phi_k[:, :, None]
                        D1_agg[:, :, e] = np.einsum('nmk,m->nk', D1_k, ex['coeffs'])
                    if need2 or need3:
                        D2_k = (gpp + gp**2) * phi_k[:, :, None]
                        D2_agg[:, :, e] = np.einsum('nmk,m->nk', D2_k, ex['coeffs'])
                    if need3:
                        D3_k = (3 * gp * gpp + gp**3) * phi_k[:, :, None]
                        D3_agg[:, :, e] = np.einsum('nmk,m->nk', D3_k, ex['coeffs'])
                else:  # wnd
                    r = np.sqrt(np.sum(diff**2 * inv2[None, :, :], axis=2))
                    rp = np.maximum(1. - r, 0.)
                    r_safe = np.maximum(r, 1e-9)
                    p1k, p2k, p3k = WND_P1(r, rp), WND_P2(r, rp), WND_P3(r, rp)
                    r0 = r_safe[:, :, None]
                    rprime = A / r0
                    rpprime = -A**2 / r0**3 + B / r0
                    if w1:
                        D1_k = p1k[:, :, None] * rprime
                        D1_agg[:, :, e] = np.einsum('nmk,m->nk', D1_k, ex['coeffs'])
                    if need2 or need3:
                        D2_k = p2k[:, :, None] * rprime**2 + p1k[:, :, None] * rpprime
                        D2_agg[:, :, e] = np.einsum('nmk,m->nk', D2_k, ex['coeffs'])
                    if need3:
                        rppprime = 3 * A * (A**2 - B * r0**2) / r0**5
                        D3_k = (p3k[:, :, None] * rprime**3
                                + 3 * p2k[:, :, None] * rprime * rpprime
                                + p1k[:, :, None] * rppprime)
                        D3_agg[:, :, e] = np.einsum('nmk,m->nk', D3_k, ex['coeffs'])
            else:  # glouton / miso
                sigma = ex['sigma']
                diff = X_H[:, None, :] - ex['centers'][None, :, :]
                sq = np.sum(diff**2, axis=2)
                phi_k = np.exp(-sq / (2 * sigma**2))
                a = np.einsum('nmd,nkd->nmk', diff, U)
                gp = -a / sigma**2
                if w1:
                    D1_k = gp * phi_k[:, :, None]
                    D1_agg[:, :, e] = np.einsum('nmk,m->nk', D1_k, ex['coeffs'])
                if need2 or need3:
                    gpp = -1 / sigma**2
                    D2_k = (gpp + gp**2) * phi_k[:, :, None]
                    D2_agg[:, :, e] = np.einsum('nmk,m->nk', D2_k, ex['coeffs'])
                if need3:
                    D3_k = (3 * gp * gpp + gp**3) * phi_k[:, :, None]
                    D3_agg[:, :, e] = np.einsum('nmk,m->nk', D3_k, ex['coeffs'])

        if w1:
            D1f = D1_agg.reshape(-1, n_exp)
            MC1_sum += D1f.T @ D1f
        if need2:
            D2f = D2_agg.reshape(-1, n_exp)
            MC2_sum += D2f.T @ D2f
        if need3:
            D3f = D3_agg.reshape(-1, n_exp)
            MC3_sum += D3f.T @ D3f
        done += K

    if w1:
        G_H += w1 * d * MC1_sum / (n_H * params_shared["n_dirs"])
    if need2:
        MC2 = MC2_sum / (n_H * params_shared["n_dirs"])
        G_H += w2 * (d * (d + 2) * MC2 - TRG_H) / 2
    if need3:
        MC3 = MC3_sum / (n_H * params_shared["n_dirs"])
        G_H += w3 * (d * (d + 2) * (d + 4) * MC3 - 9 * VVG_H) / 6

print(f" G_H calculée ({n_H} pts, {params_shared['n_dirs']} directions partagées)")

# ── Sélection des experts (même critère que précédemment) ───────────────────
n_loc = len(y_train)
m_vec = F_train.T @ y_train / n_loc
q_vec = np.sum(F_train**2, axis=0) / n_loc
denom = q_vec + params_phase2["lambda_reg"] * np.diag(G_H)
bad_denom = denom <= 1e-14
if bad_denom.any():
    print(f" [!] {bad_denom.sum()} expert(s) dégénéré(s) écarté(s)")
denom_safe = np.where(bad_denom, 1.0, denom)
expert_losses = np.mean(y_train**2) - m_vec**2 / denom_safe
expert_losses = np.where(bad_denom, np.inf, expert_losses)
best_loss = expert_losses.min()
sel = expert_losses <= params_shared["k_loss"] * best_loss
sel_idx = np.where(sel)[0]

print(f"\n Sélection experts (k_loss={params_shared['k_loss']}) : {sel.sum()}/{n_exp} retenus")
for i in range(n_exp):
    tag = "GARDÉ " if sel[i] else "écarté"
    print(f" {expert_names[i]:6s} : loss={expert_losses[i]:.3e} [{tag}]")

F_train_sel = F_train[:, sel]
F_test_sel  = F_test[:, sel]
G_H_sel     = G_H[np.ix_(sel_idx, sel_idx)]
n_sel       = sel.sum()

# ── Semi-interpolation QP (Sobolev) ─────────────────────────────────────────
# min ‖α‖_H² (+ pénalité biais)  s.t.  y_i * (F α + b)_i  ≥  qp_margin
sol_H = fit_qp_rbf(
    F_train_sel, F_test_sel, y_train, y_test, G_H_sel,
    qp_margin=params_qp["qp_margin"],
    const_pen=params_qp["const_pen"],
    lambda_G=params_qp["lambda_G"],
    thres1=params_qp["thres1"],
    thres2=params_qp["thres2"],
    titre=f"Phase2 QP ({n_sel} experts)",
)

alpha_sel = sol_H["coef"]
off_H     = sol_H["off"]
pred_H_train = sol_H["f_tr"]
pred_H_test  = sol_H["f_te"]

alpha = np.zeros(n_exp)
alpha[sel_idx] = alpha_sel

print(f"\n rang spectral Phase2 = {np.sum(np.abs(alpha_sel) > 1e-12)}/{n_sel}")
print(f" train : {cls_str(pred_H_train, y_train)}")
print(f" test  : {cls_str(pred_H_test,  y_test)}")
print(f" ‖u‖_H = {sol_H['normH']:.4f}  |  faisable={sol_H['feas']}  marge={sol_H['marge']:.3f}")

print(f"\n Poids attribués à chaque expert (alpha) :")
for i in range(n_exp):
    tag = "" if sel[i] else " (écarté, alpha=0)"
    print(f" {expert_names[i]:6s} : alpha={alpha[i]:+.4f}{tag}")

# ── Vote 1/loss (inchangé, purement informatif) ─────────────────────────────
vote = np.zeros(n_exp)
vote[sel] = 1.0 / expert_losses[sel]
vote /= vote.sum()
print("\nPoids de vote (1/loss) :")
for i in range(n_exp):
    tag = "" if sel[i] else " (écarté)"
    print(f" {expert_names[i]:6s} : vote={vote[i]:.4f}{tag}")

pred_vote_train = F_train @ vote
pred_vote_test  = F_test  @ vote

# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS (inchangées)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration Ridge polynomial...")
poly = PolynomialFeatures(degree=min(params_shared["deg_P"], 8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8)
ridge_poly.fit(poly.fit_transform(X_train), y_train)
pred_ridgepoly_te = ridge_poly.predict(poly.transform(X_test))

print("Calibration SVM RBF (GridSearchCV)...")
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(X_train, y_train)
pred_svm_te = svm.decision_function(X_test)

print("Calibration Ridge RBF (GridSearchCV)...")
param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(X_train, y_train)
pred_kr_te = kr.predict(X_test)

# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ FINAL (révisé)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("RÉSUMÉ")
print("=" * 80)
print(f"Sobolev H QP (semi-interp, {sel.sum()}/{n_exp} experts) : {cls_str(pred_H_test, y_test)}")
print(f"  → ‖u‖_H={sol_H['normH']:.4f}  faisable={sol_H['feas']}  marge={sol_H['marge']:.3f}")
print(f"Vote 1/loss                              : {cls_str(pred_vote_test, y_test)}")
print(f"Ridge polynomial                         : {cls_str(pred_ridgepoly_te, y_test)}")
print(f"SVM RBF (best={svm.best_params_})        : {cls_str(pred_svm_te, y_test)}")
print(f"Ridge RBF (best={kr.best_params_})       : {cls_str(pred_kr_te, y_test)}")

print(f"\nn_train={params_shared['n_train']}  n_unlabeled={params_shared['n_unlabeled']}  "
      f"n_test={params_shared['n_test']}  n_dirs={params_shared['n_dirs']}")
print(f"temps phase 1 : {sum(times_p1):.1f}s total ({np.mean(times_p1):.1f}s/expert)")

print("\nAccuracy/AUC individuelles des experts :")
for i in range(n_exp):
    tag = "" if sel[i] else " (écarté)"
    print(f" {expert_names[i]:6s} : {cls_str(F_test[:, i], y_test)}  loss={expert_losses[i]:.3e}{tag}")

digits [3, 5] vs [8], 7×7 -> dim 49
  train: 300  test: 5000  unlabeled: 3700
  train (+1) = 150/300
  test  (+1) = 2500/5000
Profil Wendland C4 (ordre max requis, Phase 1 wnd ∪ Phase 2 = 3)

Phase 1 (gaussien anisotrope, semi-interp) : 2 experts indépendants...
  [gauss 1/2] n_feat=800 rang=106 | faisable=True marge=1.000 ‖u‖_H=3.9750 MSE_te=34.8411 | acc=0.9222 AUC=0.9613
 gauss 1/2 | acc=0.9222 AUC=0.9613 | 54.5s
  [gauss 2/2] n_feat=800 rang=109 | faisable=True marge=1.000 ‖u‖_H=3.7996 MSE_te=62.2093 | acc=0.9196 AUC=0.9560
 gauss 2/2 | acc=0.9196 AUC=0.9560 | 43.4s

Phase 1 (Wendland, semi-interp) : 0 experts indépendants...

Phase 1 (glouton) : 1 expert (levels=9, σ0=10.0, decay=0.8)...
